# Retrieval Quality Benchmark

This notebook evaluates the quality of different retrieval strategies (Dense, BM25, Hybrid) for the MSMARCO XI dataset. Our objective is to determine the optimal configuration for semantic search.

In [ ]:
import os
import sys
import json

current_dir = os.path.abspath(os.getcwd())
colab_dir = os.path.dirname(current_dir)
if colab_dir not in sys.path:
    sys.path.append(colab_dir)

from src import utils, dataset_utils, chunking, embeddings, retrieval, evaluation

utils.set_seed(42)
repo_root = utils.find_repo_root()
config = utils.load_config()
utils.print_header("Retrieval Benchmark Initialized")

## Loading the Dataset

Before we can evaluate retrieval, we must load the dataset and extract the raw passages. This step uses the `dataset_utils` module to ensure we are working with standardized schema inputs and to fetch the MSMARCO XI documents.

In [ ]:
utils.print_header("Dataset Loading")
dataset = dataset_utils.load_msmarco_xi(config)
passages = dataset_utils.extract_passages(dataset)
print(f"Loaded {len(passages)} passages.")

## Chunking

Next, we chunk the passages. We rely on the best strategy determined from our chunking experiments (configured in `experiment_config.yaml`). The goal is to maximize semantic coherence within the chunk boundaries.

In [ ]:
utils.print_header("Chunking Strategy")
chunk_strategy = config.get('chunking', {}).get('strategy', 'sentence')
chunk_results = chunking.run_chunking(passages, chunk_strategy, config)
chunks = chunk_results.chunks
print(f"Generated {len(chunks)} chunks using {chunk_strategy} strategy.")

## Embedding Model Initialization

We initialize our embedding model to encode these chunks into a high-dimensional vector space. The dimensionality depends on the selected model and determines the FAISS index size.

In [ ]:
utils.print_header("Embedding Model Initialization")
model_name = config.get('embeddings', {}).get('model_name', 'default-model')
model = embeddings.EmbeddingModel(model_name)
print(f"Model '{model_name}' initialized with dimension {model.dimension}.")

## Corpus Embedding

We pass the chunk texts through our embedding model. This is potentially a slow, offline process that maps semantic meaning to numerical arrays.

In [ ]:
utils.print_header("Corpus Embedding")
corpus_texts = [c.text for c in chunks]
metadata_list = [{'chunk_id': getattr(c, 'id', i), 'passage_id': getattr(c, 'passage_id', i)} for i, c in enumerate(chunks)]
corpus_embeddings = model.encode(corpus_texts)
print(f"Embedded {len(corpus_embeddings)} chunks.")

## Building the Dense Index (FAISS)

With embeddings ready, we populate a FAISS Index. This allows us to perform fast exact or approximate nearest neighbor searches for dense retrieval.

In [ ]:
utils.print_header("Dense Index Creation")
dense_index = retrieval.FAISSIndex(model.dimension)
dense_index.add(corpus_embeddings, metadata_list)
print("FAISS Dense Index populated.")

## Building the Sparse Index (BM25)

To contrast semantic search, we also build a traditional BM25 inverted index based on exact term frequencies.

In [ ]:
utils.print_header("Sparse Index Creation")
bm25_index = retrieval.BM25Index()
bm25_index.add(corpus_texts, metadata_list)
print("BM25 Sparse Index populated.")

## Ground Truth Preparation & Leakage Warning

We must prepare query-document pairs to evaluate our models.

**CRITICAL WARNING:** We must avoid evaluation leakage! A query must not simply retrieve its exact original passage via trivial shortcuts. If evaluating true generalization, the query's own passage should be excluded, OR we rely purely on explicit `is_selected` labels. For this benchmark, we only consider documents where `is_selected` == 1 as relevant.

In [ ]:
utils.print_header("Preparing Evaluation Data")
eval_queries = []
for row in dataset:
    if row.get('is_selected', 0) == 1:
        eval_queries.append({'query': row['query'], 'relevant_docs': [row.get('passage_id', '')]})
print(f"Prepared {len(eval_queries)} queries for benchmarking.")

## Dense Retrieval Evaluation

We first test the dense retrieval pipeline, examining Recall@1, Recall@5, and Recall@10, as well as MRR (Mean Reciprocal Rank). Dense models often handle phrasing variations gracefully.

In [ ]:
utils.print_header("Evaluate Dense Retrieval")
dense_results = evaluation.evaluate_retrieval(eval_queries, dense_index, model, k_list=[1, 5, 10])
print("Dense Results:", dense_results.metrics)

## BM25 Retrieval Evaluation

Next, we test BM25. This sparse method struggles with synonyms but is extremely reliable when exact domain terminology is queried.

In [ ]:
utils.print_header("Evaluate BM25 Retrieval")
bm25_results = evaluation.evaluate_retrieval(eval_queries, bm25_index, None, k_list=[1, 5, 10])
print("BM25 Results:", bm25_results.metrics)

## Hybrid Retrieval (RRF)

To get the best of both worlds, we perform Reciprocal Rank Fusion (RRF). RRF merges the ranked lists from both dense and BM25 retrievers, re-scoring items based on their inverse ranks.

In [ ]:
utils.print_header("Evaluate Hybrid Retrieval")
hybrid_results = retrieval.reciprocal_rank_fusion(dense_index, bm25_index, eval_queries, model)
if not isinstance(hybrid_results, evaluation.RetrievalEvalResult):
    hybrid_results = evaluation.evaluate_retrieval(eval_queries, hybrid_results, None, k_list=[1, 5, 10])
print("Hybrid Results:", hybrid_results.metrics)

## Configuration Comparison

We compare all the configurations side-by-side using our built-in `compare_configurations` utility.

In [ ]:
utils.print_header("Comparison Table")
comparison = evaluation.compare_configurations({'Dense': dense_results, 'BM25': bm25_results, 'Hybrid': hybrid_results})
utils.print_table(comparison)

## Decision Analysis

Based on these benchmarks, the optimal retrieval strategy should balance both high exact-match accuracy and conceptual understanding. Typically, Hybrid Retrieval provides the most robust Recall@10 metrics.

In [ ]:
reports_dir = utils.get_reports_dir()
report_path = os.path.join(reports_dir, "retrieval_benchmark.json")
utils.save_json(comparison, report_path)
print(f"Successfully saved full benchmark results to: {report_path}")